In [29]:
# Palisades Wildfire — NDVI Change Analysis

# Analysis of pixel-level NDVI change by BAER burn severity and dominant vegetation type.

# Statistical methods:
# - Descriptive statistics
# - Kruskal–Wallis test
# - Epsilon-squared effect size
# - Global Moran's I

In [30]:
import os
import arcpy
import pandas as pd
import numpy as np
from scipy import stats

project_folder = arcpy.mp.ArcGISProject("CURRENT").homeFolder

results_gdb = os.path.join(
    project_folder,
    "Palisades_Wildfire_Results.gdb"
)

stats_table = os.path.join(
    results_gdb,
    "Stats_NDVI_PixelData"
)

pixel_fc = os.path.join(
    project_folder,
    "Palisades_Wildfire_Recovery.gdb",
    "Burn_Severity",
    "BAER_Pixels"
)

In [31]:
# Load the 105,006 pixel observations into a pandas DataFrame.

fields = [
    "grid_code",
    "Severity_Label",
    "NLCD_Class",
    "NLCD_Label",
    "NDVI_Change"
]

df = pd.DataFrame(
    arcpy.da.TableToNumPyArray(
        stats_table,
        fields
    )
)

print(f"{len(df):,} observations loaded.")

105,006 observations loaded.


In [32]:
# Summarize NDVI change by burn severity.

severity_summary = (
    df.groupby(
        ["grid_code", "Severity_Label"]
    )["NDVI_Change"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        StdDev="std"
    )
    .reset_index()
)

display(severity_summary)

,grid_code,Severity_Label,Count,Mean,Median,StdDev
0,1,Unburned / Very Low,3085,-0.087415,-0.082180,0.107960
1,2,Low,42287,-0.191863,-0.189504,0.119716
2,3,Moderate,56502,-0.276807,-0.285526,0.129447
3,4,High,3132,-0.387046,-0.397251,0.121654


In [33]:
# Test whether NDVI change differs among burn-severity classes.

groups = [
    df.loc[
        df["grid_code"] == code,
        "NDVI_Change"
    ]
    for code in sorted(df["grid_code"].unique())
]

kw = stats.kruskal(*groups)

n = len(df)
k = len(groups)

epsilon_squared = (
    kw.statistic - k + 1
) / (n - k)

print(f"Kruskal–Wallis H: {kw.statistic:.2f}")
print(f"p-value: {kw.pvalue:.3e}")
print(f"Epsilon-squared: {epsilon_squared:.3f}")

Kruskal–Wallis H: 18406.06
p-value: 0.000e+00
Epsilon-squared: 0.175


In [34]:
# Test the relationship between burn severity and NDVI change
# within the three dominant vegetation classes.

vegetation_codes = [42, 43, 52]

for code in vegetation_codes:

    subset = df[df["NLCD_Class"] == code]

    groups = [
        subset.loc[
            subset["grid_code"] == severity,
            "NDVI_Change"
        ]
        for severity in sorted(subset["grid_code"].unique())
    ]

    result = stats.kruskal(*groups)

    n = len(subset)
    k = len(groups)

    epsilon = (
        result.statistic - k + 1
    ) / (n - k)

    label = subset["NLCD_Label"].iloc[0]

    print(
        f"{label}: "
        f"H={result.statistic:.2f}, "
        f"p={result.pvalue:.3e}, "
        f"ε²={epsilon:.3f}"
    )

Evergreen Forest: H=1264.23, p=8.482e-274, ε²=0.160
Mixed Forest: H=2404.63, p=0.000e+00, ε²=0.077
Shrub/Scrub: H=4922.91, p=0.000e+00, ε²=0.105


In [35]:
# Evaluate whether NDVI-change values are spatially clustered.
# A 45-m neighborhood represents the immediate neighboring
# 30-m pixels, including diagonal neighbors.

moran = arcpy.stats.SpatialAutocorrelation(
    pixel_fc,
    "NDVI_Change",
    "NO_REPORT",
    "FIXED_DISTANCE_BAND",
    "EUCLIDEAN_DISTANCE",
    "ROW",
    45
)

print(f"Moran's I: {float(moran[0]):.3f}")
print(f"Z-score: {float(moran[1]):.2f}")
print(f"p-value: {float(moran[2]):.3e}")

Moran's I: 0.790
Z-score: 358.86
p-value: 0.000e+00
